# Synthetic Data Augmentation with Conditional Flow Matching

## Part 3

In this section, we evaluate the impact of incorporating synthetic image data on classification performance. In particular, we examine changes in classification accuracy and $\mathsf F_1$ score when the Fashion MNIST dataset is augmented with synthetic samples, using the full training set (i.e., 100% of the available data) for model training.

## Setup

In [1]:
!find . -mindepth 1 -exec rm -rf {} + &> /dev/null
!git clone https://github.com/ZhangLyndon/FlowMatchingAugmentation . > /dev/null 2>&1

In [2]:
!pip install -qU pip
!pip install -qU -r requirements.txt

In [3]:
import os
import sys
import argparse
import functools

# Silence tqdm output
os.environ["TQDM_DISABLE"] = "1"

# Reduce CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
import torch
import torchvision
import numpy as np

# Components for initializing an ImageNet-pretrained ResNet-18 classifier, fine-
# tuning it on Fashion MNIST, and evaluating classification performance on base-
# line, low-data, and synthetically augmented settings.
from classification import (ClassificationTrainer,
                            create_classifier, ResNetClassifier,
                            SyntheticDataGenerator, SyntheticAugmentationEvaluator,
                            create_augmented_dataset)

# Utilities for loading the Fashion MNIST dataset, computing top-k categorical
# accuracy and average cross-entropy loss, and saving training results.
from utils import get_dataloaders, AverageMeter, accuracy, save_results

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["font.family"] = "DejaVu Sans Mono"

Having fixed the training schedule at 25 epochs based on the earlier validation loss experiment to maintain a consistent evaluation pipeline, we now assess the performance impact of synthetic augmentation in the full-data regime (using the entire training set). We begin by evaluating the classification performance of the unaugmented training split, measuring both accuracy and the macro $\mathsf F_1$ score.

In [5]:
# Configure augmentation evaluation pipeline
augmentation_args = argparse.Namespace(data_root = "./data",
                                       batch_size = 16,
                                       num_workers = 0,
                                       epochs = 25,
                                       lr = 0.001,
                                       weight_decay = 1e-4,
                                       step_size = 15,
                                       gamma = 0.1,
                                       synthetic_data_dir = "./images",
                                       real_ratio = 1.0,
                                       classification_dir = "./results/classification",
                                       augmentation_dir = "./results/augmentation",
                                       checkpoint_dir = "./checkpoints",
                                       save_interval = 20,
                                       seed = 42)

# Create directory to store synthetic augmentation results
os.makedirs(augmentation_args.augmentation_dir, exist_ok = True)

# Set random seed for reproducibility
torch.manual_seed(augmentation_args.seed)
np.random.seed(augmentation_args.seed)

In [6]:
guidance_scale = 3.0
# Evaluate the ResNet classifier on the full unaugmented training set.
evaluator = SyntheticAugmentationEvaluator(augmentation_args, guidance_scale)
evaluator.run_low_data_experiments(augmentation_args.real_ratio, False)

Number of epochs: 25
Number of training samples: 60000
Number of validation samples: 10000
Epoch 1/25
Training Set | Loss: 0.6830, Top-1 Accuracy: 76.97%, Top-5 Accuracy: 98.93%
Validation Set | Loss: 0.3986, Top-1 Accuracy: 86.21%, Top-5 Accuracy: 99.71%
Best Validation Loss (Up Until Now): 0.3986
_________________________________________________________________________________________________________

Epoch 2/25
Training Set | Loss: 0.4377, Top-1 Accuracy: 85.18%, Top-5 Accuracy: 99.62%
Validation Set | Loss: 0.3543, Top-1 Accuracy: 86.89%, Top-5 Accuracy: 99.77%
Best Validation Loss (Up Until Now): 0.3543
_________________________________________________________________________________________________________

Epoch 3/25
Training Set | Loss: 0.3743, Top-1 Accuracy: 87.18%, Top-5 Accuracy: 99.67%
Validation Set | Loss: 0.3079, Top-1 Accuracy: 88.71%, Top-5 Accuracy: 99.82%
Best Validation Loss (Up Until Now): 0.3079
____________________________________________________________________

Prior to applying augmentation to the full training set, the model achieves a classification accuracy of $92.53\%$, a macro $\mathsf F_1$ score of $0.9253$, and an optimal validation (cross-entropy) loss of $0.2138$, as measured at epoch 17.